In [2]:
from pygom.model.ode_variable import ODEVariable
from pygom import InputError

import pytest
from sympy import Symbol


def test_default_real_symbol():
    ode_var = ODEVariable("x")

    assert ode_var == Symbol("x", real=True)
    assert ode_var == "x"
    assert ode_var.symbol.is_real is True

def test_complex_symbol():
    ode_var = ODEVariable("z", real = False)

    assert ode_var == Symbol("z", real=False)
    assert ode_var == "z"
    assert ode_var.symbol.is_real is False

def test_symbol():
    ode_var = ODEVariable("infectivity", symbol="beta")

    assert ode_var == Symbol("beta", real=True)
    assert ode_var == "infectivity"

def test_only_symbol():
    ode_var = ODEVariable(symbol="beta")

    assert ode_var == Symbol("beta", real=True)
    assert ode_var == "beta"

def test_vector_symbol():
    with pytest.raises(InputError):
        ODEVariable("x1:4")

def test_reject_operator():
    with pytest.raises(InputError):
        ODEVariable("x+y")

def test_reject_keyword():
    keywords = ["lambda", "if", "for"]

    for keyword in keywords:
        with pytest.raises(InputError):
            ODEVariable(keyword)

def test_reject_non_string():
    with pytest.raises(TypeError):
        ODEVariable(123)

def test_reject_leading_underscore():
    with pytest.raises(InputError):
        ODEVariable("_x")

def test_reject_leading_number():
    with pytest.raises(InputError):
        ODEVariable("1x")

test_default_real_symbol()
test_complex_symbol()
test_symbol()
test_only_symbol()
test_vector_symbol()
test_reject_operator()
test_reject_keyword()
test_reject_non_string()
test_reject_leading_number()
test_reject_leading_underscore()

In [2]:
from pygom import Transition, Event
from pygom.model.base_ode_model import BaseOdeModel

t_inf = Transition(origin="S", destination="I")

e_inf = Event(
    rate="beta*S*I/N",
    transition_list=[t_inf]
)

t_rec = Transition(origin="I", destination="R")

e_rec = Event(
    rate="gamma*I",
    transition_list=[t_rec]
)

params = ['beta', 'gamma', 'N']
states = ['S', 'I', 'R']

mod = BaseOdeModel(
    state=states,
    param=params,
    event=[e_inf, e_rec]
)

In [16]:
from sympy import parse_expr
from sympy import Expr

def checkEquation(input_str, state_param_namespace, derived_param_dict, subs_derived=True) -> list:
    """
    Convert a string into an equation using the symbols from the system and 
    checks its validity. 

    Parameters
    ----------
    input_str: a str or list of str giving the equation
    ode: the parent ode
    subs_derived: should the derived parameters be substututed in?

    Returns
    -------
    A single sympy equation or list of sympy equations (depending on if 
    input_str is a single string or a list) made from the string(s)
    """

    assert isinstance(input_str, str), "Equation should be in string format"

    eqn = parse_expr(input_str, state_param_namespace | derived_param_dict)

    if subs_derived:
        # because these are the derived parameters, we need to substitute
        # them back in the formula
        if isinstance(eqn, Expr):
            for key, value in derived_param_dict:
                eqn = eqn.subs(key, value)

    return eqn

checkEquation('beta * S * I * sin(gamma / pi)/ N', mod.states_and_parameters_dict, mod._derivedParamDict)


I*S*beta*sin(gamma/pi)/N

In [ ]:
class BaseOdeModel(object):
    """
    This base object stores the defining objects of a compartmental model
    and has functions to verify the build

    Parameters
    ----------
    state: list
        A list of states (string)
    param: list
        A list of the parameters (string)
    derived_param: list
        A list of the derived parameters (tuple of (string, string))
    transition: list
        A list of transition (:class:`.Transition`)
    event: list
        A list of events (:class:`.Transition`)
    birth_death: list
        A list of birth or death process (:class:`.Transition`)
    ode: list
        A list of ode (:class:`.Transition`)

    """
    _maths_methods = [
        EventRateVector
    ]

    def __init__(
            self,
            state=None,
            param=None,
            derived_param=None,
            event=None
        ):
        """
        Constructor
        """

        self.set_parameters(param)
        self.set_states(state)

        self.derived_param_list = derived_param
        self.event_list = event

        self._init_maths_methods()
        self._invalidate_caches()

    def set_parameters(self, parameter_list:list[str|ODEVariable]) -> None:
        """
        Set the parameters for the compartmental model

        Parameters
        ----------
        parameter_list: list
            list of strings or ode variables where each is a parameter of the 
            system
        """
        # TODO: should parameters have limits, like states?
        # create a new store to replace the existing (if creations success)
        new_parameter_store = ode_utils.ParameterStore()
        new_parameter_store.add(parameter_list)
        
        self._parameter_store = new_parameter_store
        self._invalidate_caches()

    def set_states(self, state_list:list[str|ODEVariable])->None:
        """
        Declare the states for the ode system

        Parameters
        ----------
        state_list: list
            list of strings or ode variables where each is a parameter of the 
            system
        """
        # create a new store to replace the existing (if creations succeds)
        new_state_store = ode_utils.StateStore()
        new_state_store.add(state_list)
        
        self._state_store = new_state_store
        self._invalidate_caches()

    def _init_maths_methods(self) -> None:
        """
        Add all the maths method classes as methods to this class
        """
        # Add the maths methods
        for fn_class in self._maths_methods:
            # Create an instance of the maths class with this class as the 
            # associated compartmental model system
            maths_class_instance = fn_class(parent_model=self)
            setattr(
                self,
                maths_class_instance.method_name,
                maths_class_instance
            )

            
import sympy

from .mathsmethod import NumericMethod
from .._model_verification import checkEquation

class EventRateVector(NumericMethod):
    method_name = 'event_rate_vector'
    def get_equation(self):
        """
        Get all the transitions into a vector, arranged by state to
        state transition then the birth death processes
        """

        event_rate_vector = sympy.zeros(self._parent_ode.num_events, 1)
        # Extract all info from events
        for i, event in enumerate(self._parent_ode.event_list):
            event_rate_vector[i] = checkEquation(
                event.rate,
                self._parent_ode
            )

        return event_rate_vector
import logging
import numpy as np

from .._model_errors import InputError

class MathsMethod:

    """
    TODO: how is MM different from NM?
    """

    # Should be overloaded in child classes to the method name that the class 
    # attaches.
    method_name = None 
    _cache_valid = False
    _pickleable_compile = False

    # TODO: if we try to make sure that the object is BaseODE type then we
    #       get a circular import error. I think this indicates design flaw
    # def __init__(self, parent_model: SimulateOde)->None:
    def __init__(self, parent_model)->None:
        '''
        Initialise the maths method.

        Parameters
        ----------
        parent_model: SimulateOde
            The system to which this maths method is attached
        '''
        # Save a pointer to the parent
        self._parent_model = parent_model

        # Use the parent_model's compiler class (don't want each MM having their own).
        self._SC = parent_model._SC

    def invalidate_cache(self):
        '''
        Marks the cached objects for recreation if called again
        '''
        self._cache_valid = False

    def __call__(self,):
        '''
        Dunder function so that when added to the model object it acts like a method
        
        Should be overloaded in child classes
        '''
        raise NotImplemented('This is the base class, implement this in a child class!')
    
    def get_equation(self):
        '''
        Give a symbolic form of the maths method

        Returns
        -------
        A sympy object representing the symbolic form of this method
        '''
        raise NotImplemented('This is the base class, implement this in a child class!')
    
class NumericMethod(MathsMethod):
    """
    A class designed to be attached to a model object the primary purpose is to 
    produce a numerical evaluation. The symbolic version will be compiled and 
    cached. By default you will need to provide the system state (as time and 
    state values) to perform the evaluation.
    """
    # Will store the compiled function in child classes
    _compiled_obj = None 
    _raw_fn = None

    # The type of output (matrix or vector) that the compiled expression 
    # produces, if None this will be determined automatically
    outType = None 

    def __call__(self, state, time):
        '''
        Dunder function so that when added to the model object it acts 
        like a standard method.
        
        Parameters
        ----------
        state: The values for the system states
        time: The timepoint to evaluate for
        '''
        # Check to see if we need to compile
        if not self._cache_valid or self._compiled_obj is None:
            self.compile_function()

        # perform the numerical calculation
        return self._compiled_obj(self._getEvalParam(state, time))
    
    def T(self, time, state):
        '''
        Same as :meth:`__call__` (the main method) but with time as first parameter

        This reordering is useful in the calling of integrate and similar 
        functions.
        '''
        return self.__call__(state, time)

    def compile_function(self) -> None:
        '''
        Compile the symbolic form so that rapid numerical evaluation may occur.
        Transforms the output appropriately into numpy
        '''
        logging.debug(f'Compiling sympy object {self.method_name}.')

        inputExpr = self.get_equation()

        self._raw_fn, compileType = self._SC.compileExpr(
            self._parent_model.states_and_parameters,
            inputExpr,
            backend=None,       # set at ODE level
            compileType=True    # get additional info
        )      
        
        numRow = inputExpr.rows
        numCol = inputExpr.cols

        outType = self.outType

        # define the different types of compile
        if self.outType is None:
            if numRow == 1 or numCol == 1:
                outType = "vec"
            else:
                outType = "mat"

        if outType.lower() == "vec":
            if compileType == 'np':
                self._compiled_obj = lambda x: self._raw_fn(*x).ravel()
            else:
                self._compiled_obj = lambda x: np.array(self._raw_fn(*x).tolist(),
                                                        float).ravel()
        elif outType.lower() == "mat":
            if compileType == 'np':
                self._compiled_obj = lambda x: self._raw_fn(*x)
            else:
                self._compiled_obj = lambda x: np.array(self._raw_fn(*x).tolist(), float)
        else:
            raise RuntimeError("Specified type of output not recognized")
        
        # Update the state
        self._pickleable_compile = True if self._SC._backend == 'lambda' else False
        self._cache_valid = True

    def _getEvalParam(self, state:list[float], time:float) -> list[float]:
        if state is None or time is None:
            raise InputError("Have to input both state and time")

        elif not self._parent_model._parameter_store.all_values_set:
                raise InputError("Have not set the parameters yet")

        if hasattr(state, '__iter__'):
            # just in case this isn't a list already
            eval_param = list(state) + [time]
        else:
            eval_param = [state] + [time]

        return eval_param + self._parent_model._parameter_store.values
    
    ## Funcitons  to allow pickling and unpickling
    def __getstate__(self):
        '''
        Grab the class's dict and remove the compiled objects if needed
        '''
        state = self.__dict__.copy()
        
        # Remove those compiled methods that have been added
        if not self._pickleable_compile:
            state['_compiled_obj'] = None 
            state['_raw_fn'] = None
            state['_cache_valid'] = False

        return state

# Keep as if we are going to allow pickling Cython we we need this method.    
    # def __setstate__(self, state):
    #     '''
    #     Restore the classes state with reset of compile status
    #     '''
    #     self.__dict__.update(state)

class SymbolicMethod(MathsMethod):
    """
    A class designed to be attached to a model object the primary purpose is to 
    produce a symbolic representation. The symbolic representation will be 
    cached.
    """
    _symbolic_function = None
    def __call__(self):
        '''
        Returns the symbolic representation of the method
        '''
        # Check to see if we need to compile
        if not self._cache_valid or self._symbolic_function is None:
            self._symbolic_function = self.get_equation()
            
            self._cache_valid = True
        
        return self._symbolic_function
